In [1]:
import pandas as pd
import networkx as nx
import json
from pathlib import Path
from html import escape

# Input file
csv_path = Path("../../data/transactionA.csv")

# Load CSV
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip().str.lower()

required = {"from", "to"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

# Clean addresses while preserving missing `to`
df["from"] = df["from"].astype(str).str.strip()
df["to"] = df["to"].apply(
    lambda x: None if pd.isna(x) or str(x).strip().lower() in {"", "nan", "none", "null"} else str(x).strip()
)

# Build NetworkX graph.
# MultiDiGraph preserves EVERY transaction as an individual edge.
G = nx.MultiDiGraph()

for idx, row in df.iterrows():
    sender = row["from"]
    receiver = row["to"]

    if not sender or sender.lower() == "nan":
        continue

    if receiver is None:
        # Keep contract-creation rows as metadata, not a fake NaN node.
        G.add_node(sender, node_type="address")
        continue

    G.add_node(sender, node_type="address")
    G.add_node(receiver, node_type="address")

    attrs = {
        "row": int(idx) + 1,
        "tx_hash": str(row["tx_hash"]) if "tx_hash" in df.columns and pd.notna(row["tx_hash"]) else "",
        "value": str(row["value"]) if "value" in df.columns and pd.notna(row["value"]) else "",
        "block_number": str(row["block_number"]) if "block_number" in df.columns and pd.notna(row["block_number"]) else "",
    }
    G.add_edge(sender, receiver, **attrs)

# Compute positions with NetworkX spring layout.
# This gives the browser a good initial arrangement while preserving interactivity.
pos = nx.spring_layout(G.to_undirected(), seed=42, k=1.6, iterations=120)

# Node statistics
node_degrees = dict(G.degree())
in_degrees = dict(G.in_degree())
out_degrees = dict(G.out_degree())

nodes = []
for address in G.nodes():
    short = address if len(address) <= 18 else f"{address[:8]}…{address[-6:]}"
    nodes.append({
        "id": address,
        "label": short,
        "title": f"<b>{escape(address)}</b><br>"
                 f"Transactions touching node: {node_degrees.get(address, 0)}<br>"
                 f"Incoming: {in_degrees.get(address, 0)}<br>"
                 f"Outgoing: {out_degrees.get(address, 0)}",
        "value": max(8, min(35, 8 + node_degrees.get(address, 0) * 3)),
        "x": float(pos[address][0]) * 900,
        "y": float(pos[address][1]) * 900,
    })

# Every transaction is represented separately.
edges = []
for edge_id, (u, v, key, data) in enumerate(G.edges(keys=True, data=True), start=1):
    tx_hash = data.get("tx_hash", "")
    value = data.get("value", "")
    block = data.get("block_number", "")
    row = data.get("row", "")
    tooltip = (
        f"<b>Transaction #{row}</b><br>"
        f"<b>From:</b> {escape(str(u))}<br>"
        f"<b>To:</b> {escape(str(v))}<br>"
        f"<b>Value:</b> {escape(value)}<br>"
        f"<b>Block:</b> {escape(block)}<br>"
        f"<b>TX Hash:</b> {escape(tx_hash)}"
    )
    edges.append({
        "id": edge_id,
        "from": u,
        "to": v,
        "arrows": "to",
        "title": tooltip,
        "txRow": row,
        "txHash": tx_hash,
        "value": value,
        "block": block,
    })

# Missing-to transactions
missing_to_rows = []
for idx, row in df[df["to"].isna()].iterrows():
    missing_to_rows.append({
        "row": int(idx) + 1,
        "from": str(row["from"]),
        "value": str(row["value"]) if "value" in df.columns and pd.notna(row["value"]) else "",
        "block": str(row["block_number"]) if "block_number" in df.columns and pd.notna(row["block_number"]) else "",
        "tx_hash": str(row["tx_hash"]) if "tx_hash" in df.columns and pd.notna(row["tx_hash"]) else "",
    })

data_json = json.dumps(
    {"nodes": nodes, "edges": edges, "missingTo": missing_to_rows},
    ensure_ascii=False
)

html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Ethereum Transaction Graph</title>
<script src="https://unpkg.com/vis-network/standalone/umd/vis-network.min.js"></script>
<style>
  * {{ box-sizing: border-box; }}
  body {{
    margin: 0;
    font-family: Arial, sans-serif;
    background: #f5f7fb;
    color: #18202a;
  }}
  .wrap {{
    max-width: 1500px;
    margin: 0 auto;
    padding: 18px;
  }}
  h1 {{ margin: 0 0 5px; font-size: 25px; }}
  .sub {{ color: #667085; margin-bottom: 14px; }}
  .stats {{
    display: grid;
    grid-template-columns: repeat(4, minmax(130px, 1fr));
    gap: 10px;
    margin-bottom: 12px;
  }}
  .stat {{
    background: white;
    border: 1px solid #dfe4ea;
    border-radius: 10px;
    padding: 12px;
  }}
  .stat b {{ display: block; font-size: 20px; }}
  .stat span {{ color: #667085; font-size: 12px; }}
  .controls {{
    display: flex;
    gap: 8px;
    flex-wrap: wrap;
    margin-bottom: 10px;
  }}
  button, input {{
    border: 1px solid #cfd6df;
    background: white;
    border-radius: 8px;
    padding: 9px 11px;
    font-size: 14px;
  }}
  button {{ cursor: pointer; }}
  button:hover {{ background: #f0f3f7; }}
  input {{ min-width: 280px; }}
  #network {{
    height: 690px;
    background: white;
    border: 1px solid #dfe4ea;
    border-radius: 12px;
  }}
  .panel {{
    margin-top: 12px;
    background: white;
    border: 1px solid #dfe4ea;
    border-radius: 10px;
    padding: 13px;
  }}
  .panel h3 {{ margin: 0 0 8px; }}
  #details {{
    color: #475467;
    line-height: 1.55;
    word-break: break-word;
  }}
  .warning {{
    margin-top: 10px;
    padding: 10px;
    background: #fff8e6;
    border: 1px solid #f0d38a;
    border-radius: 8px;
  }}
  @media (max-width: 700px) {{
    .stats {{ grid-template-columns: repeat(2, 1fr); }}
    #network {{ height: 600px; }}
    input {{ min-width: 200px; flex: 1; }}
  }}
</style>
</head>
<body>
<div class="wrap">
  <h1>Ethereum Transaction Network</h1>
  <div class="sub">Interactive NetworkX-generated layout • Every valid transaction is preserved as an individual directed edge</div>

  <div class="stats">
    <div class="stat"><b id="txCount">0</b><span>Transaction rows</span></div>
    <div class="stat"><b id="nodeCount">0</b><span>Unique addresses</span></div>
    <div class="stat"><b id="edgeCount">0</b><span>Graph edges</span></div>
    <div class="stat"><b id="missingCount">0</b><span>Missing <code>to</code></span></div>
  </div>

  <div class="controls">
    <input id="search" type="text" placeholder="Search address or TX hash…">
    <button id="searchBtn">Find</button>
    <button id="resetBtn">Reset view</button>
    <button id="physicsBtn">Toggle physics</button>
    <button id="labelsBtn">Toggle labels</button>
    <button id="fitBtn">Fit graph</button>
  </div>

  <div id="network"></div>

  <div class="panel">
    <h3>Selected node / transaction</h3>
    <div id="details">Click a wallet/address or an edge to inspect it.</div>
    <div id="missingBox" class="warning"></div>
  </div>
</div>

<script>
const DATA = {data_json};

const nodeData = new vis.DataSet(DATA.nodes);
const edgeData = new vis.DataSet(DATA.edges);

const container = document.getElementById("network");

const options = {{
  physics: {{
    enabled: false,
    stabilization: {{iterations: 100}}
  }},
  interaction: {{
    hover: true,
    navigationButtons: true,
    keyboard: true,
    multiselect: false,
    zoomView: true,
    dragView: true
  }},
  nodes: {{
    shape: "dot",
    borderWidth: 1,
    font: {{size: 10, face: "Arial"}},
    scaling: {{min: 8, max: 35}}
  }},
  edges: {{
    smooth: {{type: "dynamic"}},
    arrows: {{to: {{enabled: true, scaleFactor: 0.7}}}},
    width: 1,
    color: {{opacity: 0.45}},
    selectionWidth: 3
  }}
}};

const network = new vis.Network(
  container,
  {{nodes: nodeData, edges: edgeData}},
  options
);

document.getElementById("txCount").textContent = DATA.nodes.length
  ? DATA.edges.length + DATA.missingTo.length : 0;
document.getElementById("nodeCount").textContent = DATA.nodes.length;
document.getElementById("edgeCount").textContent = DATA.edges.length;
document.getElementById("missingCount").textContent = DATA.missingTo.length;

const missingBox = document.getElementById("missingBox");
if (DATA.missingTo.length) {{
  missingBox.innerHTML =
    "<b>Missing <code>to</code> transactions:</b> " +
    DATA.missingTo.length +
    " row(s). They are intentionally not connected to a fake NaN node. " +
    "They may represent contract-creation transactions and need receipt data to identify the created contract.";
}} else {{
  missingBox.style.display = "none";
}}

network.on("click", function(params) {{
  const details = document.getElementById("details");

  if (params.nodes.length) {{
    const id = params.nodes[0];
    const node = nodeData.get(id);
    const degree = nodeData.get(id) ? (
      network.getConnectedEdges(id).length
    ) : 0;

    details.innerHTML =
      "<b>Address:</b> " + id + "<br>" +
      "<b>Connected transaction edges:</b> " + degree + "<br>" +
      "<b>Incoming transactions:</b> " +
      DATA.edges.filter(e => e.to === id).length + "<br>" +
      "<b>Outgoing transactions:</b> " +
      DATA.edges.filter(e => e.from === id).length;
    return;
  }}

  if (params.edges.length) {{
    const e = edgeData.get(params.edges[0]);
    details.innerHTML =
      "<b>Transaction row:</b> #" + e.txRow + "<br>" +
      "<b>From:</b> " + e.from + "<br>" +
      "<b>To:</b> " + e.to + "<br>" +
      "<b>Value:</b> " + (e.value || "N/A") + "<br>" +
      "<b>Block:</b> " + (e.block || "N/A") + "<br>" +
      "<b>TX hash:</b> " + (e.txHash || "N/A");
    return;
  }}

  details.textContent = "Click a wallet/address or an edge to inspect it.";
}});

document.getElementById("resetBtn").addEventListener("click", () => {{
  network.unselectAll();
  network.fit({{animation: true}});
  document.getElementById("details").textContent =
    "Click a wallet/address or an edge to inspect it.";
}});

let physicsOn = false;
document.getElementById("physicsBtn").addEventListener("click", () => {{
  physicsOn = !physicsOn;
  network.setOptions({{physics: {{enabled: physicsOn}}}});
}});

let labelsOn = true;
document.getElementById("labelsBtn").addEventListener("click", () => {{
  labelsOn = !labelsOn;
  nodeData.forEach(n => {{
    nodeData.update({{id: n.id, label: labelsOn ? n.label : ""}});
  }});
}});

document.getElementById("fitBtn").addEventListener("click", () => {{
  network.fit({{animation: true}});
}});

function findItem() {{
  const q = document.getElementById("search").value.trim().toLowerCase();
  if (!q) return;

  const node = DATA.nodes.find(n => n.id.toLowerCase() === q || n.id.toLowerCase().includes(q));
  if (node) {{
    network.selectNodes([node.id]);
    network.focus(node.id, {{
      scale: 1.4,
      animation: {{duration: 700, easingFunction: "easeInOutQuad"}}
    }});
    return;
  }}

  const edge = DATA.edges.find(e =>
    (e.txHash || "").toLowerCase().includes(q) ||
    String(e.txRow).toLowerCase() === q
  );

  if (edge) {{
    network.selectEdges([edge.id]);
    network.focus(edge.from, {{
      scale: 1.2,
      animation: {{duration: 700, easingFunction: "easeInOutQuad"}}
    }});
    document.getElementById("details").innerHTML =
      "<b>Transaction row:</b> #" + edge.txRow + "<br>" +
      "<b>From:</b> " + edge.from + "<br>" +
      "<b>To:</b> " + edge.to + "<br>" +
      "<b>TX hash:</b> " + (edge.txHash || "N/A");
    return;
  }}

  document.getElementById("details").textContent = "No matching address or transaction found.";
}}

document.getElementById("searchBtn").addEventListener("click", findItem);
document.getElementById("search").addEventListener("keydown", e => {{
  if (e.key === "Enter") findItem();
}});
</script>
</body>
</html>
"""

output = Path("/mnt/data/ethereum_transaction_graph_interactive.html")
output.write_text(html, encoding="utf-8")

print(f"Created: {output}")
print(f"Transactions: {len(df)}")
print(f"Unique addresses: {G.number_of_nodes()}")
print(f"Address-to-address transactions: {G.number_of_edges()}")
print(f"Missing 'to': {len(missing_to_rows)}")


TypeError: object of type 'float' has no len()